# Lenguajes y Autómatas II (SCD-1016)
## Sesión 08 · Inferencia y Promoción de Tipos

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué tipo tiene el resultado de mezclar un entero con un flotante?
>
> **Objetivo de la sesión:** Aplicar la jerarquía de promoción para decidir el tipo de una operación y distinguir cuándo una conversión pierde información.

**Entregable:** tus reglas en artesano_sesion08_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Implementas promover(a,b) y hay_perdida(destino,origen); el harness mezcla tipos y comprueba la jerarquía.

Al final de la sesión tendrás: **Una función que decide el tipo común de dos operandos y otra que avisa si una conversión pierde datos.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 08](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion08.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion08.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion08.h"

Tipo promover(Tipo a, Tipo b);
int hay_perdida(Tipo destino, Tipo origen);
const char *nombre_tipo(Tipo t);
```

- Jerarquía: CHAR < INT < FLOAT.
- promover devuelve el tipo más ancho; si uno es ERROR, devuelve ERROR.
- hay_perdida es 1 cuando el destino es más angosto que el origen.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion08_c.c` junto con `artesano_sesion08_c.c`
> (o `ia_sesion08_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `promover(CHAR,INT)` | `int` |
| 2 | `promover(INT,FLOAT)` | `float` |
| 3 | `promover(FLOAT,FLOAT)` | `float` |
| 4 | `promover(ERROR,INT)` | `error` |
| 5 | `hay_perdida(CHAR,INT)` | 1 |
| 6 | `hay_perdida(INT,CHAR)` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 08 - Reglas de promocion y perdida ===
Caso                               esperado:              obtenido:              resultado
[ 1] promover(CHAR,INT)                 esperado: int          obtenido: int          OK
[ 2] promover(INT,FLOAT)                esperado: float        obtenido: float        OK
[ 3] promover(FLOAT,FLOAT)              esperado: float        obtenido: float        OK
[ 4] promover(ERROR,INT)                esperado: error        obtenido: error        OK
[ 5] hay_perdida(CHAR,INT)              esperado: 1            obtenido: 1            OK
[ 6] hay_perdida(INT,CHAR)              esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion08.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] promover devuelve siempre el tipo de mayor rango y nunca pierde el centinela ERROR.
- [ ] hay_perdida solo es cierto cuando el destino es más angosto.
- [ ] Los seis casos del harness salen OK con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion08_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion08.h
/* api_sesion08.h - contrato de la Sesion 08
 * Reglas de promoción y pérdida
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION08_H
#define API_SESION08_H

typedef enum { TIPO_ERROR = 0, TIPO_CHAR = 1, TIPO_INT = 2, TIPO_FLOAT = 3 } Tipo;

Tipo promover(Tipo a, Tipo b);
int hay_perdida(Tipo destino, Tipo origen);
const char *nombre_tipo(Tipo t);

#endif /* API_SESION08_H */


In [ ]:
%%writefile artesano_sesion08_c.c
/* artesano_sesion08_c.c - Fase 1: El Artesano (sin IA)
 * Reglas de promoción y pérdida.
 *
 * Implementa lo que declara api_sesion08.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion08.h"
#include <stdio.h>
#include <string.h>

Tipo promover(Tipo a, Tipo b) {
    (void)a; (void)b; /* TODO: devuelve el tipo mas ancho */
    return TIPO_ERROR;
}

int hay_perdida(Tipo destino, Tipo origen) {
    (void)destino; (void)origen; /* TODO: compara los rangos */
    return 0;
}

const char *nombre_tipo(Tipo t) {
    (void)t; /* TODO: nombre legible */
    return "desconocido";
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion08_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion08_c.c
/* ia_sesion08_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion08.h. No puede traer main().
 */
#include "api_sesion08.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion08_c.c
/* harness_sesion08_c.c - auditoria automatica de la Sesion 08
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion08_c.c artesano_sesion08_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion08.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 08 - Reglas de promocion y perdida ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    {  verificar_str("promover(CHAR,INT)", "int", nombre_tipo(promover(TIPO_CHAR, TIPO_INT))); }
    {  verificar_str("promover(INT,FLOAT)", "float", nombre_tipo(promover(TIPO_INT, TIPO_FLOAT))); }
    {  verificar_str("promover(FLOAT,FLOAT)", "float", nombre_tipo(promover(TIPO_FLOAT, TIPO_FLOAT))); }
    {  verificar_str("promover(ERROR,INT)", "error", nombre_tipo(promover(TIPO_ERROR, TIPO_INT))); }
    {  verificar_int("hay_perdida(CHAR,INT)", (long)(1), (long)(hay_perdida(TIPO_CHAR, TIPO_INT))); }
    {  verificar_int("hay_perdida(INT,CHAR)", (long)(0), (long)(hay_perdida(TIPO_INT, TIPO_CHAR))); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion08.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion08_c.c artesano_sesion08_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion08_c.c ia_sesion08_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion08.md` a `auditorias/auditoria_sesion08.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
